# Cooperative Inverse Reinforcement Learning for Portfolio Management

**A Toy Study in Online Preference Learning**

### Problem

We consider a single risky asset with Gaussian returns and cash (risk-free, zero interest). The human's preferences are parameterized by θ = (λ_risk, λ_turn), where λ_risk penalizes return variance (risk aversion) and λ_turn penalizes trading activity (transaction costs or behavioral friction). The human makes decisions according to a Boltzmann-rational model: they are more likely to choose actions with higher utility, but not perfectly deterministic.

### CIRL
- both the agent and the human cooperate to achieve the human's goals
- the agent maintains a Bayesian belief over the human's preference parameters θ
- occasionally, the agent queries the human by presenting pairwise comparisons between candidate trading actions.
- the human's choice provides information about θ
- the agent updates its belief via Bayes' rule. 
- between queries, the agent trades greedily according to the current posterior mean estimate θ̂.
- model-based control with online Bayesian inference over the reward function
- **objective**: minimize regret (the gap between the agent's total reward and the oracle's total reward who knows true θ* at the start), minimize the number of queries used by the agent to get to oracle-grade performance
- testing: we compare the CIRL-based agent to a baseline that guesses θ or learns it offline



In [2]:
# Import dependencies
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Any

# Set random seed for reproducibility
np.random.seed(42)

#### Environment 

- **State**: `[price, shares, cash]`
  - `price`: Current price of the risky asset
  - `shares`: Number of shares held
  - `cash`: Cash holdings

- **Dynamics**: The risky asset has Gaussian returns:
  - r_t ~ Normal(μ, σ)
  - price_{t+1} = price_t × (1 + r_t)

- **Actions**: Discrete set {-1, 0, +1}
  - -1: Sell one share
  - 0: Hold
  - +1: Buy one share

- **Transaction Costs**: A small proportional cost applies to trades:
  - cost = trans_cost × |action| × price

- **Rewards**: return the change in portfolio value (PnL) as the raw reward, which will later be transformed according to the human's preferences.

This environment can be terminates in T steps.

In [7]:
class PortfolioEnv:
    """
    State: [price, shares, cash]
    Actions: {-1, 0, +1} (sell, hold, buy)
    Dynamics: Gaussian returns with transaction costs
    """
    
    def __init__(self, mu: float = 0.0005, sigma: float = 0.01, 
                 T: int = 100, trans_cost: float = 0.001,
                 init_price: float = 100.0, init_cash: float = 1000.0):
        """
            mu: Mean return of the risky asset per time step
            sigma: Standard deviation of returns
            T: Episode horizon (number of time steps)
            trans_cost: Proportional transaction cost (e.g., 0.001 = 0.1%)
            init_price: Initial price of the risky asset
            init_cash: Initial cash holdings
        """
        self.mu = mu
        self.sigma = sigma
        self.T = T
        self.trans_cost = trans_cost
        self.init_price = init_price
        self.init_cash = init_cash
        
        # State variables
        self.price = None
        self.shares = None
        self.cash = None
        self.t = None
        
    def reset(self) -> np.ndarray:
        """
        Reset the environment -> initial state.
        
        Returns:
            Initial state [price, shares, cash]
        """
        self.price = self.init_price
        self.shares = 0.0
        self.cash = self.init_cash
        self.t = 0
        return self._get_state()
    
    def _get_state(self) -> np.ndarray:
        return np.array([self.price, self.shares, self.cash])
    
    def _portfolio_value(self) -> float:
        return self.shares * self.price + self.cash
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict[str, Any]]:
        """
        One time step of the environment.
        """
        value_before = self._portfolio_value()
        
        cost = self.trans_cost * abs(action) * self.price
        self.cash -= cost
        
        trade_value = action * self.price
        self.shares += action
        self.cash -= trade_value
        
        if self.shares < 0 or self.cash < 0:
            # Revert trade if invalid
            self.shares -= action
            self.cash += trade_value
            self.cash += cost
            action = 0  # no op
        
        return_t = np.random.normal(self.mu, self.sigma)
        self.price = self.price * (1 + return_t)
        
        # PnL (change in portfolio value)
        value_after = self._portfolio_value()
        raw_pnl = value_after - value_before
        
        # Update time step
        self.t += 1
        done = (self.t >= self.T)
        
        info = {
            'time': self.t,
            'portfolio_value': value_after,
            'transaction_cost': cost,
            'return': return_t,
            'actual_action': action
        }
        
        return self._get_state(), raw_pnl, done, info

    def copy(self) -> 'PortfolioEnv':
        """
        Creates a copy of the environment
        """
        env_copy = PortfolioEnv(
            mu=self.mu, sigma=self.sigma, T=self.T, 
            trans_cost=self.trans_cost, init_price=self.init_price, 
            init_cash=self.init_cash
        )
        env_copy.price = self.price
        env_copy.shares = self.shares
        env_copy.cash = self.cash
        env_copy.t = self.t
        return env_copy

In [8]:
print('example environment set')

env = PortfolioEnv(mu=0.0005, sigma=0.01, T=10)
state = env.reset()

total_pnl = 0
# 5 steps example

for i in range(5):
    action = np.random.choice([-1, 0, 1])
    next_state, raw_pnl, done, info = env.step(action)
    total_pnl += raw_pnl
    
    print(f"Step {i+1}:")
    print(f"  Action: {action:+d} ({'Sell' if action == -1 else 'Hold' if action == 0 else 'Buy'})")
    print(f"  State: [price={next_state[0]:.2f}, shares={next_state[1]:.1f}, cash={next_state[2]:.2f}]")
    print(f"  Raw PnL: {raw_pnl:+.2f}")
    print(f"  Portfolio value: {info['portfolio_value']:.2f}")
    print(f"  Transaction cost: {info['transaction_cost']:.4f}")

example environment set
Step 1:
  Action: +1 (Buy)
  State: [price=101.06, shares=1.0, cash=899.90]
  Raw PnL: +0.96
  Portfolio value: 1000.96
  Transaction cost: 0.1000
Step 2:
  Action: +0 (Hold)
  State: [price=101.19, shares=1.0, cash=899.90]
  Raw PnL: +0.13
  Portfolio value: 1001.09
  Transaction cost: 0.0000
Step 3:
  Action: +0 (Hold)
  State: [price=101.08, shares=1.0, cash=899.90]
  Raw PnL: -0.11
  Portfolio value: 1000.98
  Transaction cost: 0.0000
Step 4:
  Action: +0 (Hold)
  State: [price=102.61, shares=1.0, cash=899.90]
  Raw PnL: +1.53
  Portfolio value: 1002.51
  Transaction cost: 0.0000
Step 5:
  Action: -1 (Sell)
  State: [price=104.24, shares=0.0, cash=1002.41]
  Raw PnL: -0.10
  Portfolio value: 1002.41
  Transaction cost: 0.1026


#### Human Pref. Model

##### Reward Function

**R(PnL, action; θ) = PnL - λ_risk × PnL² - λ_turn × 𝟙[action ≠ 0]**

- **PnL**: Raw profit/loss from the action
- **λ_risk**: Risk penalty (penalizes variance in returns)
- **λ_turn**: Turnover penalty (discourages frequent trading)

The quadratic risk term (PnL²) encourages smoother, more consistent returns. The turnover term penalizes any non-zero action.

##### Ground-Truth Preferences

θ* = (λ_risk*, λ_turn*) - the human's true preferences. 

##### Human Choice Model

The human chooses actions (given some candidates) according to a **Boltzmann-rational** model which simulates a tendency towards better actions but occasionally they make suboptimal choices:

**P(action | candidates, state) ∝ exp(β × Q(state, action))**

- Q(state, action): expected utility of taking that action (approximated using the environment's expected return μ)
- β: an inverse temperature parameter controlling rationality. 
- Higher β -> the human is more deterministic; Lower β -> more random.


In [9]:
def trading_reward(raw_pnl: float, action: int, theta: np.ndarray) -> float:
    lambda_risk, lambda_turn = theta
    
    risk_penalty = lambda_risk * (raw_pnl ** 2)
    
    turnover_penalty = lambda_turn * (1 if action != 0 else 0)
    
    reward = raw_pnl - risk_penalty - turnover_penalty
    return reward

# human preferences
theta_star = np.array([0.1, 0.1])

print("Ground-truth preference:")
print(f"λ_risk*: {theta_star[0]}")
print(f"λ_turn*: {theta_star[1]} ")

Ground-truth preference:
λ_risk*: 0.1
λ_turn*: 0.1 


In [10]:
def compute_expected_q(state: np.ndarray, action: int, theta: np.ndarray, 
                       env: PortfolioEnv) -> float:
    """
    Expected Q-value for an action using deterministic lookahead.
    """
    # current state
    env_copy = env.copy()
    
    value_before = env_copy._portfolio_value()
    
    cost = env_copy.trans_cost * abs(action) * env_copy.price
    env_copy.cash -= cost
    
    trade_value = action * env_copy.price
    env_copy.shares += action
    env_copy.cash -= trade_value
    
    # Check for invalid trades
    if env_copy.shares < 0 or env_copy.cash < 0:

        return -1e6

    env_copy.price = env_copy.price * (1 + env_copy.mu)

    value_after = env_copy._portfolio_value()
    expected_pnl = value_after - value_before
    
    q_value = trading_reward(expected_pnl, action, theta)
    
    return q_value


def softmax_probabilities(q_values: np.ndarray, beta: float = 5.0) -> np.ndarray:
    """
    Compute softmax probabilities from Q-values (distribution over actions)
    """
    q_max = np.max(q_values)
    exp_q = np.exp(beta * (q_values - q_max))
    probs = exp_q / np.sum(exp_q)
    return probs


def human_choice(state: np.ndarray, candidates: List[int], 
                 theta_star: np.ndarray, env: PortfolioEnv, 
                 beta: float = 5.0) -> int:
    """
    Simulate a human's choice
    """
    q_values = np.array([
        compute_expected_q(state, action, theta_star, env)
        for action in candidates
    ])
    probs = softmax_probabilities(q_values, beta)
    chosen_idx = np.random.choice(len(candidates), p=probs)
    chosen_action = candidates[chosen_idx]
    
    return chosen_action

In [11]:
# Test human choice model
env_test = PortfolioEnv()
state_test = env_test.reset()

candidates_test = [-1, 0, 1]
print(f"\nState: [price={state_test[0]:.2f}, shares={state_test[1]:.1f}, cash={state_test[2]:.2f}]")
print(f"Candidates: {candidates_test}")
print(f"True preferences: λ_risk={theta_star[0]}, λ_turn={theta_star[1]}")

print("Expected Q-values under true preferences:")
for action in candidates_test:
    q_val = compute_expected_q(state_test, action, theta_star, env_test)
    print(f"Action {action:+d}: Q = {q_val:.4f}")

print("Simulate 1000 human choices (β=4.0):")
choices = [human_choice(state_test, candidates_test, theta_star, env_test, beta=4.0) 
           for _ in range(1000)]
for action in candidates_test:
    count = choices.count(action)
    print(f"Action {action:+d}: {count/10:.1f}% of choices")


State: [price=100.00, shares=0.0, cash=1000.00]
Candidates: [-1, 0, 1]
True preferences: λ_risk=0.1, λ_turn=0.1
Expected Q-values under true preferences:
Action -1: Q = -1000000.0000
Action +0: Q = 0.0000
Action +1: Q = -0.1503
Simulate 1000 human choices (β=4.0):
Action -1: 0.0% of choices
Action +0: 66.3% of choices
Action +1: 33.7% of choices
